# Tabelas globais por objetivo de otimização

Este notebook consolida, para o modelo `Llama3.1-I`, comparações entre:
- `lod`;
- `llama / initial_system_prompt` (resultado `without_optimization`);
- `llama / best_system_prompt` (melhor configuração `with_optimization` para o objetivo analisado).

A saída é organizada por objetivo de otimização:
- `sep`
- `etd`
- `sep_etd_f1`

Para cada objetivo, o notebook:
1. escolhe o melhor processo otimizado por algoritmo com base na métrica-alvo do próprio objetivo;
2. mostra uma tabela-resumo da melhor configuração selecionada;
3. exibe três tabelas com as métricas `SEP`, `ETD` e `SEP_ETD_F1` para `lod`, `initial_system_prompt` e `best_system_prompt`;
4. anota diretamente os valores de `best_system_prompt` com dois símbolos por célula, na ordem `vs initial` e depois `vs lod`, obtidos por teste de Wilcoxon pareado:
   - `▲` para melhor e significativo,
   - `●` para ausência de diferença significativa,
   - `▼` para pior e significativo.


In [1]:
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

try:
    from scipy.stats import wilcoxon
except ImportError:
    wilcoxon = None


In [2]:
def _is_project_root(candidate: Path) -> bool:
    return (candidate / "run_prompt_optimizer.py").exists() and (candidate / "out").exists()


def find_local_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if _is_project_root(candidate):
            return candidate

    descendant_hints = []
    for base in (start, *start.parents):
        descendant_hints.extend(
            [
                base / "prompt-optim-expl-rec" / "explainability-with-LLMs",
                base / "explainability-with-LLMs",
            ]
        )

    for candidate in descendant_hints:
        if _is_project_root(candidate):
            return candidate

    raise FileNotFoundError(
        "Não foi possível localizar a raiz de explainability-with-LLMs. "
        "Execute o notebook no projeto, em um subdiretório dele ou a partir da raiz do workspace."
    )

PROJECT_ROOT = find_local_project_root(Path.cwd().resolve())
MODEL_NAME = "Llama3.1-I"
SUMMARY_CSV = PROJECT_ROOT / "out" / "statistical_analysis" / "common" / "per_user_run_summary.csv"
STATISTICAL_ANALYSIS_ROOT = PROJECT_ROOT / "out" / "statistical_analysis"
LOD_ROOT = PROJECT_ROOT / "out" / "results_lod"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"SUMMARY_CSV: {SUMMARY_CSV}")
print(f"STATISTICAL_ANALYSIS_ROOT: {STATISTICAL_ANALYSIS_ROOT}")
print(f"LOD_ROOT: {LOD_ROOT}")


PROJECT_ROOT: /mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_WebMedia/origin/06_prompt-optimization/prompt-optim-expl-rec/explainability-with-LLMs
SUMMARY_CSV: /mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_WebMedia/origin/06_prompt-optimization/prompt-optim-expl-rec/explainability-with-LLMs/out/statistical_analysis/common/per_user_run_summary.csv
STATISTICAL_ANALYSIS_ROOT: /mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_WebMedia/origin/06_prompt-optimization/prompt-optim-expl-rec/explainability-with-LLMs/out/statistical_analysis
LOD_ROOT: /mnt/ssd/prenassi/OTIMIAZAO_RECOMENDACAO/pos_WebMedia/origin/06_prompt-optimization/prompt-optim-expl-rec/explainability-with-LLMs/out/results_lod


In [3]:
OBJECTIVE_ORDER = ["sep", "etd", "sep_etd_f1"]
ALGORITHM_ORDER = ["user_knn", "item_knn", "bprmf", "ncf"]
TARGET_COLUMN_BY_OBJECTIVE = {
    "sep": "mean_sep_per_user",
    "etd": "mean_etd_per_user",
    "sep_etd_f1": "mean_sep_etd_f1_per_user",
}
DISPLAY_METRICS = [
    ("mean_sep_per_user", "sep_value", "SEP", "sep"),
    ("mean_etd_per_user", "etd_value", "ETD", "etd"),
    ("mean_sep_etd_f1_per_user", "sep_etd_f1_value", "SEP_ETD_F1", "sep_etd_f1"),
]
ALPHA = 0.05
SYMBOLS = {
    "gain": "▲",
    "tie": "●",
    "loss": "▼",
    "missing": "-",
}
ROW_INDEX = [
    ("lod", ""),
    ("llama", "initial_system_prompt"),
    ("llama", "best_system_prompt"),
]


def ordered_algorithms(values: list[str]) -> list[str]:
    ordered = [algorithm for algorithm in ALGORITHM_ORDER if algorithm in values]
    ordered.extend(sorted(algorithm for algorithm in values if algorithm not in ordered))
    return ordered


def harmonic_mean(left: float, right: float) -> float:
    if pd.isna(left) or pd.isna(right) or (left + right) == 0:
        return 0.0
    return float((2.0 * left * right) / (left + right))


def load_best_wide_df(statistical_analysis_root: Path, objective_metric: str) -> pd.DataFrame:
    path = (
        statistical_analysis_root
        / objective_metric
        / "results"
        / f"{objective_metric}_best_method_by_algorithm_per_user_wide.csv"
    )
    if not path.exists():
        return pd.DataFrame()

    df = pd.read_csv(path)
    score_columns = [
        "sep_lod",
        "etd_lod",
        "sep_etd_f1_lod",
        "sep_llama_without_optimization",
        "etd_llama_without_optimization",
        "sep_etd_f1_llama_without_optimization",
        "sep_llama_with_optimization",
        "etd_llama_with_optimization",
        "sep_etd_f1_llama_with_optimization",
    ]
    for column in score_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    return df


def build_best_wide_by_objective(statistical_analysis_root: Path) -> dict[str, pd.DataFrame]:
    return {
        objective_metric: load_best_wide_df(statistical_analysis_root, objective_metric)
        for objective_metric in OBJECTIVE_ORDER
    }


def wilcoxon_symbol_from_series(
    best_series: pd.Series,
    reference_series: pd.Series,
    alpha: float = ALPHA,
) -> str:
    paired = pd.DataFrame(
        {
            "best": pd.to_numeric(best_series, errors="coerce"),
            "reference": pd.to_numeric(reference_series, errors="coerce"),
        }
    ).dropna()

    if paired.empty or wilcoxon is None:
        return SYMBOLS["missing"]

    try:
        result = wilcoxon(
            paired["best"],
            paired["reference"],
            alternative="two-sided",
            zero_method="wilcox",
        )
        p_value = float(result.pvalue)
    except ValueError:
        return SYMBOLS["missing"]

    if pd.isna(p_value) or p_value >= alpha:
        return SYMBOLS["tie"]

    mean_best = float(paired["best"].mean())
    mean_reference = float(paired["reference"].mean())
    if mean_best > mean_reference:
        return SYMBOLS["gain"]
    if mean_best < mean_reference:
        return SYMBOLS["loss"]
    return SYMBOLS["tie"]


def load_summary(summary_csv: Path) -> pd.DataFrame:
    if not summary_csv.exists():
        raise FileNotFoundError(f"Resumo estatístico não encontrado em {summary_csv}")

    df = pd.read_csv(summary_csv)
    numeric_cols = [
        "mean_sep_per_user",
        "mean_etd_per_user",
        "mean_sep_etd_f1_per_user",
        "n_users",
    ]
    for column in numeric_cols:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    return df


def load_lod_summary(lod_root: Path) -> pd.DataFrame:
    rows = []
    for csv_path in sorted(lod_root.glob("indiv_metrics_explanations_optimized_*_K=20_recs.csv.xls")):
        algorithm = (
            csv_path.name
            .replace("indiv_metrics_explanations_optimized_", "")
            .replace("_K=20_recs.csv.xls", "")
        )
        frame = pd.read_csv(csv_path)
        frame["sep"] = pd.to_numeric(frame["sep"], errors="coerce")
        frame["etd"] = pd.to_numeric(frame["etd"], errors="coerce")
        frame["sep_etd_f1"] = frame.apply(
            lambda row: harmonic_mean(row["sep"], row["etd"]),
            axis=1,
        )
        rows.append(
            {
                "algorithm": algorithm,
                "mean_sep_per_user": frame["sep"].mean(),
                "mean_etd_per_user": frame["etd"].mean(),
                "mean_sep_etd_f1_per_user": frame["sep_etd_f1"].mean(),
                "n_users": int(frame["userId"].nunique()),
                "source_group": "lod",
                "source_label": "",
                "representation_model": pd.NA,
                "mmr_lambda_tag": pd.NA,
                "run_label": f"lod/{algorithm}",
            }
        )

    if not rows:
        return pd.DataFrame(
            columns=[
                "algorithm",
                "mean_sep_per_user",
                "mean_etd_per_user",
                "mean_sep_etd_f1_per_user",
                "n_users",
                "source_group",
                "source_label",
                "representation_model",
                "mmr_lambda_tag",
                "run_label",
            ]
        )

    return pd.DataFrame(rows)


def format_repr_model(value: str | float | None) -> str:
    if value is None or pd.isna(value):
        return "-"
    return str(value).replace("repr_", "")


def format_lambda(value: str | float | None) -> str:
    if value is None or pd.isna(value):
        return "-"
    return str(value).replace("mmr_lambda_", "").replace("_", ".")


def pick_initial_rows(summary_df: pd.DataFrame, objective_metric: str) -> pd.DataFrame:
    frame = summary_df[
        (summary_df["run_type"] == "without_optimization")
        & (summary_df["metric"] == objective_metric)
    ].copy()
    frame["source_group"] = "llama"
    frame["source_label"] = "initial_system_prompt"
    return frame


def pick_best_optimized_rows(summary_df: pd.DataFrame, objective_metric: str) -> pd.DataFrame:
    target_column = TARGET_COLUMN_BY_OBJECTIVE[objective_metric]
    frame = summary_df[
        (summary_df["run_type"] == "with_optimization")
        & (summary_df["metric"] == objective_metric)
    ].copy()

    if frame.empty:
        return frame

    frame = frame.sort_values(
        by=[
            "algorithm",
            target_column,
            "mean_sep_etd_f1_per_user",
            "mean_sep_per_user",
            "mean_etd_per_user",
            "run_label",
        ],
        ascending=[True, False, False, False, False, True],
        na_position="last",
    )
    frame = frame.drop_duplicates(subset=["algorithm"], keep="first")
    frame["source_group"] = "llama"
    frame["source_label"] = "best_system_prompt"
    return frame


def build_objective_comparison_df(
    summary_df: pd.DataFrame,
    lod_df: pd.DataFrame,
    objective_metric: str,
) -> pd.DataFrame:
    frames = []

    initial_df = pick_initial_rows(summary_df, objective_metric)
    if not initial_df.empty:
        frames.append(initial_df)

    best_df = pick_best_optimized_rows(summary_df, objective_metric)
    if not best_df.empty:
        frames.append(best_df)

    if not lod_df.empty:
        lod_view = lod_df.copy()
        lod_view["metric"] = objective_metric
        frames.append(lod_view)

    if not frames:
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined["objective_metric"] = objective_metric
    combined["algorithm"] = pd.Categorical(
        combined["algorithm"],
        categories=ordered_algorithms(combined["algorithm"].dropna().astype(str).unique().tolist()),
        ordered=True,
    )
    return combined.sort_values(by=["algorithm", "source_group", "source_label"]).reset_index(drop=True)


def build_metric_pivot(
    comparison_df: pd.DataFrame,
    metric_column: str,
    top_label: str,
) -> pd.DataFrame:
    pivot = comparison_df.pivot_table(
        index=["source_group", "source_label"],
        columns="algorithm",
        values=metric_column,
        aggfunc="first",
        observed=False,
    )
    pivot = pivot.reindex(
        index=pd.MultiIndex.from_tuples(ROW_INDEX, names=[None, None])
    )
    algorithms = ordered_algorithms([str(column) for column in pivot.columns.tolist()])
    pivot = pivot.reindex(columns=algorithms)
    pivot.columns = pd.MultiIndex.from_product([[top_label], pivot.columns])
    return pivot


def build_significance_symbol_map(
    best_wide_df: pd.DataFrame,
    metric_key: str,
) -> dict[str, dict[str, str]]:
    if best_wide_df.empty:
        return {}

    best_column = f"{metric_key}_llama_with_optimization"
    initial_column = f"{metric_key}_llama_without_optimization"
    lod_column = f"{metric_key}_lod"
    required_columns = {"algorithm", best_column, initial_column, lod_column}
    if not required_columns.issubset(best_wide_df.columns):
        return {}

    available_algorithms = ordered_algorithms(
        best_wide_df["algorithm"].dropna().astype(str).unique().tolist()
    )
    symbols_by_algorithm = {}

    for algorithm in available_algorithms:
        symbols_by_algorithm[algorithm] = {
            "best_vs_initial": wilcoxon_symbol_from_series(
                best_series=best_wide_df.loc[best_wide_df["algorithm"] == algorithm, best_column],
                reference_series=best_wide_df.loc[best_wide_df["algorithm"] == algorithm, initial_column],
            ),
            "best_vs_lod": wilcoxon_symbol_from_series(
                best_series=best_wide_df.loc[best_wide_df["algorithm"] == algorithm, best_column],
                reference_series=best_wide_df.loc[best_wide_df["algorithm"] == algorithm, lod_column],
            )
        }
    return symbols_by_algorithm


def build_annotated_metric_table(
    metric_table: pd.DataFrame,
    best_wide_df: pd.DataFrame,
    metric_key: str,
    top_label: str,
) -> pd.DataFrame:
    if metric_table.empty:
        return pd.DataFrame()

    values_df = metric_table[top_label].copy()
    annotated_df = values_df.copy().astype(object)
    for row_index in annotated_df.index:
        for algorithm in annotated_df.columns:
            value = values_df.loc[row_index, algorithm]
            annotated_df.loc[row_index, algorithm] = "-" if pd.isna(value) else f"{float(value):.6f}"

    symbols_by_algorithm = build_significance_symbol_map(
        best_wide_df=best_wide_df,
        metric_key=metric_key,
    )

    best_index = ("llama", "best_system_prompt")
    if best_index in annotated_df.index:
        for algorithm in annotated_df.columns:
            base_value = annotated_df.loc[best_index, algorithm]
            if base_value == "-":
                continue

            symbol_info = symbols_by_algorithm.get(
                str(algorithm),
                {"best_vs_initial": SYMBOLS["missing"], "best_vs_lod": SYMBOLS["missing"]},
            )
            best_vs_initial = symbol_info["best_vs_initial"]
            best_vs_lod = symbol_info["best_vs_lod"]
            if best_vs_initial == SYMBOLS["missing"] and best_vs_lod == SYMBOLS["missing"]:
                continue

            annotated_df.loc[best_index, algorithm] = (
                f"{best_vs_initial}{best_vs_lod} {base_value}"
            )

    annotated_df.columns = pd.MultiIndex.from_product([[top_label], annotated_df.columns])
    return annotated_df


def build_best_config_table(comparison_df: pd.DataFrame) -> pd.DataFrame:
    best_df = comparison_df[comparison_df["source_label"] == "best_system_prompt"].copy()
    if best_df.empty:
        return pd.DataFrame()

    best_df["representation_model"] = best_df["representation_model"].apply(format_repr_model)
    best_df["mmr_lambda_quality"] = best_df["mmr_lambda_tag"].apply(format_lambda)

    columns = [
        "algorithm",
        "representation_model",
        "mmr_lambda_quality",
        "mean_sep_per_user",
        "mean_etd_per_user",
        "mean_sep_etd_f1_per_user",
        "run_label",
    ]
    return best_df[columns].rename(
        columns={
            "algorithm": "algorithm",
            "representation_model": "representation_model",
            "mmr_lambda_quality": "mmr_lambda_quality",
            "mean_sep_per_user": "sep_value",
            "mean_etd_per_user": "etd_value",
            "mean_sep_etd_f1_per_user": "sep_etd_f1_value",
            "run_label": "optimized_run_label",
        }
    ).reset_index(drop=True)


In [4]:
summary_df = load_summary(SUMMARY_CSV)
lod_df = load_lod_summary(LOD_ROOT)
best_wide_by_objective = build_best_wide_by_objective(STATISTICAL_ANALYSIS_ROOT)

if summary_df.empty:
    warnings.warn("O resumo por usuário está vazio. Nenhuma tabela foi montada.")
else:
    display(Markdown(
        "As tabelas abaixo usam, para cada objetivo, o melhor processo otimizado por algoritmo "
        "segundo a própria métrica-alvo desse objetivo."
    ))
    display(Markdown(
        "Os símbolos aparecem na linha `best_system_prompt`, na ordem `vs initial` e depois `vs lod`. "
        "Legenda do Wilcoxon: `▲` = melhor e significativo, "
        "`●` = sem diferença significativa e `▼` = pior e significativo."
    ))

    for objective_metric in OBJECTIVE_ORDER:
        comparison_df = build_objective_comparison_df(
            summary_df=summary_df,
            lod_df=lod_df,
            objective_metric=objective_metric,
        )

        if comparison_df.empty:
            display(Markdown(f"## Objetivo `{objective_metric}`"))
            display(Markdown("> Nenhum dado encontrado para este objetivo."))
            continue

        display(Markdown(f"## Objetivo `{objective_metric}`"))
        display(Markdown("### Melhor configuração otimizada por algoritmo"))
        best_config_df = build_best_config_table(comparison_df)
        best_wide_df = best_wide_by_objective.get(objective_metric, pd.DataFrame())
        if best_config_df.empty:
            display(Markdown("> Nenhuma configuração otimizada foi encontrada."))
        else:
            display(best_config_df.style.format(
                {
                    "sep_value": "{:.6f}",
                    "etd_value": "{:.6f}",
                    "sep_etd_f1_value": "{:.6f}",
                },
                na_rep="-",
            ))

        for metric_column, top_label, metric_label, metric_key in DISPLAY_METRICS:
            display(Markdown(
                f"### Tabela de `{metric_label}` para processos otimizados por `{objective_metric}`"
            ))
            metric_table = build_metric_pivot(
                comparison_df=comparison_df,
                metric_column=metric_column,
                top_label=top_label,
            )
            annotated_metric_table = build_annotated_metric_table(
                metric_table=metric_table,
                best_wide_df=best_wide_df,
                metric_key=metric_key,
                top_label=top_label,
            )
            display(annotated_metric_table)


As tabelas abaixo usam, para cada objetivo, o melhor processo otimizado por algoritmo segundo a própria métrica-alvo desse objetivo.

Os símbolos aparecem na linha `best_system_prompt`, na ordem `vs initial` e depois `vs lod`. Legenda do Wilcoxon: `▲` = melhor e significativo, `●` = sem diferença significativa e `▼` = pior e significativo.

## Objetivo `sep`

### Melhor configuração otimizada por algoritmo

,algorithm,representation_model,mmr_lambda_quality,sep_value,etd_value,sep_etd_f1_value,optimized_run_label
0,user_knn,sbert,0.5,0.710932,0.701639,0.685187,with_optimization/Llama3.1-I/user_knn/sep/repr_sbert/early_false/mmr_lambda_0_5/mmr_pool_10
1,item_knn,llm2vec,0.5,0.701002,0.663934,0.662004,with_optimization/Llama3.1-I/item_knn/sep/repr_llm2vec/early_false/mmr_lambda_0_5/mmr_pool_10
2,bprmf,llm2vec,0.0,0.711800,0.713934,0.694968,with_optimization/Llama3.1-I/bprmf/sep/repr_llm2vec/early_false/mmr_lambda_0_0/mmr_pool_10
3,ncf,sbert,0.5,0.730987,0.646721,0.664689,with_optimization/Llama3.1-I/ncf/sep/repr_sbert/early_false/mmr_lambda_0_5/mmr_pool_10


...

sep_value                            \
algorithm                       user_knn     item_knn        bprmf   
lod                             0.596438     0.560192     0.608610   
llama initial_system_prompt     0.658011     0.626747     0.662111   
      best_system_prompt     ▲▲ 0.692547  ●▲ 0.621315  ▲▲ 0.685957   

                                          
algorithm                            ncf  
lod                             0.605711  
llama initial_system_prompt     0.628054  
      best_system_prompt     ●● 0.619523

### Tabela de `ETD` para processos otimizados por `sep_etd_f1`

etd_value                            \
algorithm                       user_knn     item_knn        bprmf   
lod                             0.509016     0.580328     0.524590   
llama initial_system_prompt     0.757377     0.757377     0.785246   
      best_system_prompt     ●▲ 0.748273  ▲▲ 0.795082  ●▲ 0.771311   

                                          
algorithm                            ncf  
lod                             0.559016  
llama initial_system_prompt     0.776230  
      best_system_prompt     ▲▲ 0.818852

### Tabela de `SEP_ETD_F1` para processos otimizados por `sep_etd_f1`

sep_etd_f1_value                            \
algorithm                           user_knn     item_knn        bprmf   
lod                                 0.522916     0.547630     0.536263   
llama initial_system_prompt         0.689348     0.671721     0.702121   
      best_system_prompt         ●▲ 0.700532  ●▲ 0.678642  ●▲ 0.712135   

                                          
algorithm                            ncf  
lod                             0.559040  
llama initial_system_prompt     0.675460  
      best_system_prompt     ●▲ 0.691292